# Demonstração dos padrões do livro

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/demonstracao.ipynb)

Esta página mostra, num exemplo pequeno, os quatro tipos de figura que aparecem em todos os capítulos. O exemplo é o método de Newton para $f(x) = x^2 - 2$, cuja raiz é $\sqrt{2}$.

1. Uma figura que explica a **ideia** do método.
2. Um gráfico de **convergência**, que verifica a teoria.
3. Uma **comparação** entre métodos.
4. Uma figura com **controle embutido** (slider), com a sua versão estática para o PDF.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

## 1. A ideia em uma imagem

O método de Newton troca a curva pela reta tangente no ponto atual e vai até onde essa reta corta o eixo $x$. Esse ponto é o próximo palpite:

$$
x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}
$$

In [ ]:
def f(x):
    return x**2 - 2

def df(x):
    return 2 * x

def figura_newton(x0, n_passos=3, titulo=None):
    """Desenha a curva e as tangentes das primeiras iterações de Newton."""
    r = raizes.newton(f, df, x0, max_iter=n_passos)
    xs = np.linspace(0.2, 3.2, 300)
    fig = go.Figure()
    fig.add_hline(y=0, line=dict(color="#444444", width=1))
    fig.add_trace(go.Scatter(x=xs, y=f(xs), name="f(x) = x² − 2", line=dict(color=CORES["exata"])))
    for k, (xk, xk1) in enumerate(zip(r.historico[:-1], r.historico[1:])):
        fig.add_trace(go.Scatter(x=[xk, xk1], y=[f(xk), 0], mode="lines",
                                 line=dict(color=CORES["aproximacao"], dash="dot"),
                                 name="tangente" if k == 0 else None, showlegend=(k == 0)))
        fig.add_trace(go.Scatter(x=[xk, xk], y=[0, f(xk)], mode="lines",
                                 line=dict(color=CORES["referencia"], width=1), showlegend=False))
    # Rótulos só nos pontos que não se sobrepõem aos anteriores.
    sub = "₀₁₂₃₄₅₆₇₈₉"
    rotulos, ultimo = [], None
    for k, xk in enumerate(r.historico):
        livre = ultimo is None or abs(xk - ultimo) > 0.15
        rotulos.append(f"x{sub[k]}" if livre else "")
        ultimo = xk if livre else ultimo
    fig.add_trace(go.Scatter(x=r.historico, y=[0] * len(r.historico), mode="markers+text",
                             text=rotulos, textposition="bottom center",
                             hovertemplate="x = %{x:.6f}<extra></extra>",
                             marker=dict(color=CORES["destaque"], size=9), name="iterações"))
    fig.update_xaxes(title="x", range=[0.2, 3.2])
    fig.update_yaxes(title="f(x)", range=[-3, 8])
    fig.update_layout(title=titulo or f"Newton a partir de x₀ = {numero(x0)}")
    return fig

mostrar(figura_newton(3.0))

**O que notar:** em três passos, as tangentes já chegam tão perto de $\sqrt{2} \approx 1{,}414$ que os pontos se sobrepõem. Passe o mouse sobre a figura para ver os valores.

## 2. Verificação: o erro segue a teoria?

Este é o gráfico mais importante do livro. Ele mostra o erro em escala log-log, ao lado das retas de **ordem teórica**. Se os pontos ficam paralelos à reta de ordem $p$, o método se comporta como a teoria prevê.

No exemplo, a derivada de $\sin x$ em $x = 1$ é aproximada por diferenças progressivas (ordem 1) e centrais (ordem 2).

In [ ]:
h = np.logspace(-1, -4, 10)
x0 = 1.0
erro_prog = np.abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0))
erro_cent = np.abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2 * h) - np.cos(x0))

fig = grafico_convergencia(h, erro_cent, ordens=(1, 2), nome="diferença central",
                           titulo="Erro da derivada numérica de sin(x) em x = 1")
fig.add_trace(go.Scatter(x=h, y=erro_prog, mode="lines+markers", name="diferença progressiva",
                         line=dict(color=CORES["secundaria"])))
mostrar(fig)

print("Ordem observada, progressiva:", numero(ordem_observada(h, erro_prog), 2))
print("Ordem observada, central:    ", numero(ordem_observada(h, erro_cent), 2))

**O que notar:** a ordem medida (1,00 e 2,00) confirma a análise de Taylor. É esse tipo de prova que o aluno entrega quando o código foi escrito por um agente.

## 3. Comparação entre métodos

Todas as funções de `cnlivro.raizes` devolvem o histórico das iterações. Isso permite comparar métodos lado a lado, com o mesmo gráfico.

In [ ]:
resultados = {
    "bisseção": raizes.bissecao(f, 0, 3),
    "falsa posição": raizes.falsa_posicao(f, 0, 3),
    "secante": raizes.secante(f, 3.0, 2.5),
    "Newton": raizes.newton(f, df, 3.0),
}
mostrar(grafico_iteracoes(resultados, titulo="Quatro métodos para x² − 2 = 0"))

for nome, r in resultados.items():
    print(f"{nome:14s} {r.iteracoes:3d} iterações")
print("Ordem medida de Newton:", numero(ordem_iterativa(resultados["Newton"].historico, np.sqrt(2)), 2))

**O que notar:** a bisseção reduz o erro pela metade a cada passo (uma reta no gráfico). Newton dobra o número de dígitos corretos a cada passo (uma curva que despenca).

## 4. Explore

A figura abaixo tem um **controle embutido**: arraste o slider para mudar o palpite inicial $x_0$. Ela funciona direto no site, sem Python rodando. No PDF aparece o estado inicial da figura, seguido de uma versão estática com três valores fixos de $x_0$.

In [ ]:
valores_x0 = np.round(np.arange(0.3, 3.01, 0.1), 1)
quadros = [(numero(v), figura_newton(v, 3)) for v in valores_x0]
titulos = [q[1].layout.title.text for q in quadros]
mostrar(com_controle(quadros, rotulo="x₀ = ", ativo=len(valores_x0) - 1, titulos=titulos))

In [ ]:
# Versão estática (pequenos múltiplos) do controle acima, usada no PDF.
valores = [0.5, 1.0, 3.0]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f"x₀ = {numero(v)}" for v in valores])
for j, v in enumerate(valores, start=1):
    for tr in figura_newton(v, 3).data:
        tr.showlegend = (j == 1) and tr.showlegend is not False
        fig.add_trace(tr, row=1, col=j)
fig.update_xaxes(range=[0.2, 3.2], title="x")
fig.update_yaxes(range=[-3, 8])
fig.update_yaxes(title="f(x)", col=1)
fig.update_layout(width=900, height=400, margin=dict(t=90), title="Newton a partir de três palpites iniciais")
mostrar(fig)

**O que notar:** com $x_0 = 0{,}5$ a primeira tangente é quase horizontal e manda o palpite para longe, mas o método ainda volta. No capítulo 3, o leitor vai procurar palpites em que isso *não* acontece.